In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/workspace/sic_final_project")
needed = ["data/processed/images_64.npy", "data/processed/labels.npy",
          "data/splits/split_seed42.csv", "data/metadata/metadata_clean.csv"]
for rel in needed:
    print(f"{'OK     ' if (PROJECT_ROOT / rel).exists() else 'MISSING'} {rel}")

for d in ["src", "models/traditional", "results/figures", "results/metrics", "results/comparisons"]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

split_df = pd.read_csv(PROJECT_ROOT / "data/splits/split_seed42.csv")
ok = split_df["path"].map(lambda p: Path(p).exists()).sum()
print(f"Image paths in split file that exist: {ok} / {len(split_df)}")

gi = PROJECT_ROOT / ".gitignore"
if not gi.exists():
    gi.write_text("Cats and Dogs Classification Dataset/\n*.zip\ndata/processed/\nmodels/\n"
                  "*.npy\n*.joblib\n*.pth\n*.pt\n.ipynb_checkpoints/\n__pycache__/\n")
    print(".gitignore created")
else:
    print(".gitignore already exists (left unchanged)")

In [ ]:
%%writefile /workspace/sic_final_project/src/evaluation.py
"""Shared evaluation helpers (used by all phases)."""
import time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

CLASS_NAMES = ["Cat", "Dog"]      # label 0 = Cat, label 1 = Dog. Dog is the "positive" class.
PROJECT_ROOT = Path(__file__).resolve().parents[1]          # works wherever the project folder lives
RESULTS_CSV = PROJECT_ROOT / "results/comparisons/model_results.csv"


def compute_metrics(y_true, y_pred, y_score):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, y_score)}


def log_result(experiment, split, metrics, **extra):
    """Insert or replace the row (experiment, split) in the shared results table."""
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    row = pd.DataFrame([{"experiment": experiment, "split": split, **metrics, **extra}])
    if RESULTS_CSV.exists():
        old = pd.read_csv(RESULTS_CSV)
        old = old[~((old["experiment"] == experiment) & (old["split"] == split))]
        row = pd.concat([old, row], ignore_index=True)
    row.to_csv(RESULTS_CSV, index=False)


def evaluate_sklearn(model, X, y):
    """Predict, score and time a fitted scikit-learn model. Returns metrics, labels, scores, ms/image."""
    t0 = time.perf_counter()
    y_pred = model.predict(X)
    ms_per_img = 1000 * (time.perf_counter() - t0) / len(X)
    score = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)
    return compute_metrics(y, y_pred, score), y_pred, score, ms_per_img


def plot_confusion(y_true, y_pred, title, ax=None):
    cm = confusion_matrix(y_true, y_pred)
    ax = ax or plt.gca()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
    return cm

In [ ]:
%%writefile /workspace/sic_final_project/src/traditional_models.py
"""CPU scikit-learn models and feature helpers for flattened-image features (Phases 5-6)."""
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier


def flatten(images_uint8):
    """(N, H, W, C) uint8 -> (N, H*W*C) float32 scaled to [0, 1]."""
    return images_uint8.reshape(len(images_uint8), -1).astype(np.float32) / 255.0


def make_majority():
    return DummyClassifier(strategy="most_frequent")


def make_rf(n_estimators=100):
    return RandomForestClassifier(n_estimators=n_estimators, n_jobs=8, random_state=42)


def make_svm_exact(C=1.0):
    """Exact libsvm RBF SVM (slow). Used only to sanity-check the GPU approximation on a subset."""
    return make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma="scale", cache_size=4000))

In [ ]:
%%writefile /workspace/sic_final_project/src/gpu_models.py
"""GPU (PyTorch) versions of the Phase-5/6 classical models.
fit() takes float32 tensors X (N, D) and integer labels y (N,) in {0, 1} on the same device.
score_dog(X) returns a NumPy array; a sample is predicted Dog when score > self.threshold."""
import math
import numpy as np
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class GPUStandardScaler:
    """Per-feature standardisation. Statistics come ONLY from the data given to fit() (the train set)."""
    def fit(self, X):
        self.mean = X.mean(dim=0)
        self.std = X.std(dim=0).clamp_min(1e-6)
        return self

    def transform(self, X):
        return (X - self.mean) / self.std


def _lbfgs(params, loss_fn, max_iter):
    """Full-batch L-BFGS with strong-Wolfe line search. Returns the final loss value."""
    opt = torch.optim.LBFGS(params, lr=1.0, max_iter=max_iter, max_eval=int(max_iter * 1.5),
                            history_size=20, tolerance_grad=1e-7, tolerance_change=1e-10,
                            line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = loss_fn()
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        return float(loss_fn())


class GPULogisticRegression:
    """L2-regularised logistic regression with the scikit-learn objective
    C * sum(log-loss) + 0.5 * ||w||^2 (intercept not penalised), divided by C*N."""
    threshold = 0.5

    def __init__(self, C=1e-3, max_iter=300):
        self.C, self.max_iter = C, max_iter

    def fit(self, X, y):
        self.scaler = GPUStandardScaler().fit(X)
        Xs = self.scaler.transform(X)
        n, d = Xs.shape
        yf = y.float()
        w = torch.zeros(d, device=X.device, requires_grad=True)
        b = torch.zeros(1, device=X.device, requires_grad=True)
        lam = 1.0 / (2.0 * self.C * n)

        def loss_fn():
            return F.binary_cross_entropy_with_logits(Xs @ w + b, yf) + lam * (w * w).sum()

        self.final_loss = _lbfgs([w, b], loss_fn, self.max_iter)
        self.w, self.b = w.detach(), b.detach()
        return self

    @torch.no_grad()
    def score_dog(self, X):
        return torch.sigmoid(self.scaler.transform(X) @ self.w + self.b).cpu().numpy()


class GPUKNN:
    """Exact brute-force KNN, Euclidean distance on the raw 0-1 pixels (no scaling)."""
    threshold = 0.5

    def __init__(self, k=5, chunk=512):
        self.k, self.chunk = k, chunk

    def fit(self, X, y):
        self.X, self.y = X, y.float()
        self.sq = (X * X).sum(dim=1)
        return self

    @torch.no_grad()
    def neighbours(self, X, k_max):
        out = []
        for s in range(0, len(X), self.chunk):
            q = X[s:s + self.chunk]
            d2 = (q * q).sum(dim=1, keepdim=True) + self.sq[None, :] - 2.0 * (q @ self.X.T)
            out.append(d2.topk(k_max, dim=1, largest=False).indices)
        return torch.cat(out)

    @torch.no_grad()
    def score_dog(self, X):
        return self.y[self.neighbours(X, self.k)].mean(dim=1).cpu().numpy()


class GPURBFSVM:
    """APPROXIMATE RBF-kernel SVM: random Fourier features (Rahimi & Recht) + a linear L2-SVM
    (squared hinge loss) trained with L-BFGS on the GPU. Not identical to libsvm's exact SVC."""
    threshold = 0.0

    def __init__(self, C=1.0, n_components=4096, gamma="scale", max_iter=300, seed=42):
        self.C, self.D, self.gamma, self.max_iter, self.seed = C, n_components, gamma, max_iter, seed

    def _phi(self, Xs):
        return math.sqrt(2.0 / self.D) * torch.cos(Xs @ self.W + self.off)

    def fit(self, X, y):
        self.scaler = GPUStandardScaler().fit(X)
        Xs = self.scaler.transform(X)
        n, d = Xs.shape
        # gamma="scale" is scikit-learn's rule: 1 / (n_features * variance of the scaled data)
        self.gamma_ = 1.0 / (d * float(Xs.var())) if self.gamma == "scale" else float(self.gamma)
        g = torch.Generator().manual_seed(self.seed)             # CPU generator, reproducible anywhere
        self.W = (torch.randn(d, self.D, generator=g) * math.sqrt(2.0 * self.gamma_)).to(X.device)
        self.off = (torch.rand(self.D, generator=g) * 2.0 * math.pi).to(X.device)
        Z = self._phi(Xs)
        del Xs
        ypm = y.float() * 2.0 - 1.0                               # labels {0,1} -> {-1,+1}
        w = torch.zeros(self.D, device=X.device, requires_grad=True)
        b = torch.zeros(1, device=X.device, requires_grad=True)
        lam = 1.0 / (2.0 * self.C * n)

        def loss_fn():
            margin = (1.0 - ypm * (Z @ w + b)).clamp_min(0.0)
            return (margin * margin).mean() + lam * (w * w).sum()

        self.final_loss = _lbfgs([w, b], loss_fn, self.max_iter)
        self.w, self.b = w.detach(), b.detach()
        return self

    @torch.no_grad()
    def score_dog(self, X):
        return (self._phi(self.scaler.transform(X)) @ self.w + self.b).cpu().numpy()

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "8"
import sys, json, time, random, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, torch
from pathlib import Path
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(8)
PROJECT_ROOT = Path("/workspace/sic_final_project")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
MODEL_DIR = PROJECT_ROOT / "models/traditional"

import evaluation, traditional_models, gpu_models
for m in (evaluation, traditional_models, gpu_models): importlib.reload(m)
print("Results table:", evaluation.RESULTS_CSV)

split_df = pd.read_csv(PROJECT_ROOT / "data/splits/split_seed42.csv")
imgs64 = np.load(PROJECT_ROOT / "data/processed/images_64.npy", mmap_mode="r")
labels = np.load(PROJECT_ROOT / "data/processed/labels.npy")

tr = split_df.loc[split_df["split"] == "train", "array_index"].to_numpy()
va = split_df.loc[split_df["split"] == "val", "array_index"].to_numpy()
# The test split is deliberately NOT loaded in this notebook.
assert (labels[tr] == split_df.loc[split_df["split"] == "train", "label_id"].to_numpy()).all()

X_train, y_train = traditional_models.flatten(imgs64[tr]), labels[tr]
X_val, y_val = traditional_models.flatten(imgs64[va]), labels[va]
print("Q30 - one 64x64 RGB image has", 64 * 64 * 3, "features")
print("X_train", X_train.shape, X_train.dtype, f"{X_train.nbytes/1e9:.2f} GB | X_val", X_val.shape)
print("train classes:", np.bincount(y_train), "| val classes:", np.bincount(y_val))

In [ ]:
dev = gpu_models.DEVICE
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "CPU ONLY - tell me, the GPU is not visible")
Xg_train = torch.from_numpy(X_train).to(dev)
yg_train = torch.from_numpy(y_train).to(dev)
Xg_val = torch.from_numpy(X_val).to(dev)
print(f"on device: train {tuple(Xg_train.shape)}, val {tuple(Xg_val.shape)} | "
      f"allocated {torch.cuda.memory_allocated()/1e9:.2f} GB")

sub = np.random.default_rng(SEED).choice(len(X_train), 3000, replace=False)   # fixed subset for train-accuracy checks
sub_t = torch.from_numpy(sub).to(dev)
val_preds, fitted = {}, {}

def _sync():
    if dev.type == "cuda": torch.cuda.synchronize()
_ = Xg_val[:64] @ Xg_val[:64].T; _sync()          # warm-up so CUDA start-up isn't counted in timings

def val_acc(model):
    return float(((model.score_dog(Xg_val) > model.threshold).astype(int) == y_val).mean())

def run_gpu_experiment(name, model, notes="", train_acc=True):
    _sync(); t0 = time.perf_counter(); model.fit(Xg_train, yg_train); _sync(); fit_s = time.perf_counter() - t0
    _sync(); t0 = time.perf_counter(); score = model.score_dog(Xg_val); _sync()
    ms = 1000 * (time.perf_counter() - t0) / len(Xg_val)
    y_pred = (score > model.threshold).astype(int)
    metrics = evaluation.compute_metrics(y_val, y_pred, score)
    tr_acc = np.nan
    if train_acc:
        s_tr = model.score_dog(Xg_train[sub_t])
        tr_acc = float(((s_tr > model.threshold).astype(int) == y_train[sub]).mean())
    evaluation.log_result(name, "val", metrics, train_accuracy=tr_acc, train_time_s=round(fit_s, 1),
                          infer_ms_per_img=round(ms, 4), notes=notes + " | GPU (PyTorch)")
    val_preds[name], fitted[name] = (y_pred, score), model
    print(f"{name}: val acc {metrics['accuracy']:.4f} | train acc (3k) {tr_acc:.4f} | AUC {metrics['roc_auc']:.4f} | "
          f"fit {fit_s:.1f}s | {ms:.4f} ms/img")

def run_cpu_experiment(name, model, notes=""):
    t0 = time.perf_counter(); model.fit(X_train, y_train); fit_s = time.perf_counter() - t0
    metrics, y_pred, score, ms = evaluation.evaluate_sklearn(model, X_val, y_val)
    tr_acc = float(model.score(X_train[sub], y_train[sub]))
    evaluation.log_result(name, "val", metrics, train_accuracy=tr_acc, train_time_s=round(fit_s, 1),
                          infer_ms_per_img=round(ms, 4), notes=notes + " | CPU (scikit-learn)")
    val_preds[name], fitted[name] = (y_pred, score), model
    print(f"{name}: val acc {metrics['accuracy']:.4f} | train acc (3k) {tr_acc:.4f} | AUC {metrics['roc_auc']:.4f} | "
          f"fit {fit_s:.1f}s | {ms:.4f} ms/img")

run_cpu_experiment("Majority baseline", traditional_models.make_majority(), notes="predicts the most common training class")

In [ ]:
lr_rows = []
for C in [1e-5, 1e-4, 1e-3, 1e-2]:               # smaller C = stronger regularisation
    _sync(); t0 = time.perf_counter()
    m = gpu_models.GPULogisticRegression(C).fit(Xg_train, yg_train); _sync()
    lr_rows.append({"C": C, "val_acc": val_acc(m), "final_loss": round(m.final_loss, 4),
                    "fit_s": round(time.perf_counter() - t0, 1)})
display(pd.DataFrame(lr_rows))
best_C = max(lr_rows, key=lambda r: r["val_acc"])["C"]
run_gpu_experiment("Logistic Regression", gpu_models.GPULogisticRegression(best_C), notes=f"C={best_C}, L-BFGS")

In [ ]:
knn = gpu_models.GPUKNN().fit(Xg_train, yg_train)
_sync(); t0 = time.perf_counter()
idx = knn.neighbours(Xg_val, 41); _sync()          # neighbours found once, reused for every k
print(f"neighbour search for {len(Xg_val)} val images: {time.perf_counter()-t0:.1f}s")
votes = yg_train[idx].float()
ks = [1, 3, 5, 11, 21, 41]                          # odd values only, so there are no ties
knn_rows = [{"k": k, "val_acc": float(((votes[:, :k].mean(1) > 0.5).cpu().numpy().astype(int) == y_val).mean())} for k in ks]
display(pd.DataFrame(knn_rows))
best_k = max(knn_rows, key=lambda r: r["val_acc"])["k"]
run_gpu_experiment("KNN", gpu_models.GPUKNN(k=best_k), notes=f"k={best_k}, Euclidean on 0-1 pixels", train_acc=False)

In [ ]:
svm_rows = []
for C in [0.1, 1.0, 10.0]:
    _sync(); t0 = time.perf_counter()
    m = gpu_models.GPURBFSVM(C=C, n_components=4096).fit(Xg_train, yg_train); _sync()
    svm_rows.append({"C": C, "D": 4096, "val_acc": val_acc(m), "gamma": round(m.gamma_, 6), "fit_s": round(time.perf_counter() - t0, 1)})
best_svm_C = max(svm_rows, key=lambda r: r["val_acc"])["C"]

# does a richer kernel approximation (more random features) help?
_sync(); t0 = time.perf_counter()
m = gpu_models.GPURBFSVM(C=best_svm_C, n_components=8192).fit(Xg_train, yg_train); _sync()
svm_rows.append({"C": best_svm_C, "D": 8192, "val_acc": val_acc(m), "gamma": round(m.gamma_, 6), "fit_s": round(time.perf_counter() - t0, 1)})
display(pd.DataFrame(svm_rows))
best_svm_D = 8192 if svm_rows[-1]["val_acc"] > max(r["val_acc"] for r in svm_rows[:-1]) else 4096
del m; torch.cuda.empty_cache()

run_gpu_experiment("SVM - Raw Pixels", gpu_models.GPURBFSVM(C=best_svm_C, n_components=best_svm_D),
                   notes=f"APPROX RBF SVM: random Fourier features D={best_svm_D} + linear L2-SVM, C={best_svm_C}, gamma=scale")

In [ ]:
run_cpu_experiment("Random Forest", traditional_models.make_rf(100), notes="100 trees, max_features=sqrt, fully grown")

In [ ]:
table = pd.read_csv(evaluation.RESULTS_CSV)
order = ["Majority baseline", "Logistic Regression", "KNN", "SVM - Raw Pixels", "Random Forest"]
table = table[table["experiment"].isin(order)].set_index("experiment")
table = table.loc[[o for o in order if o in table.index]]
display(table.round(4))

names = [n for n in order if n in val_preds]
fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4))
for ax, n in zip(np.atleast_1d(axes), names):
    cm = evaluation.plot_confusion(y_val, val_preds[n][0], n, ax=ax)
    print(f"{n:22s} errors: {cm[0,1] + cm[1,0]}  (Cat->Dog {cm[0,1]}, Dog->Cat {cm[1,0]})")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/05_traditional_confusion.png", dpi=110); plt.show()

for n in names[1:]:
    print("\n" + n); print(classification_report(y_val, val_preds[n][0], target_names=evaluation.CLASS_NAMES, digits=4))
    

In [ ]:
n_chk = 3000
ci = np.random.default_rng(SEED + 2).choice(len(X_train), n_chk, replace=False)
ci_t = torch.from_numpy(ci).to(dev)
t0 = time.perf_counter()
exact = traditional_models.make_svm_exact(1.0).fit(X_train[ci], y_train[ci])
t_exact = time.perf_counter() - t0
acc_exact = float((exact.predict(X_val) == y_val).mean())
approx = gpu_models.GPURBFSVM(C=1.0, n_components=4096).fit(Xg_train[ci_t], yg_train[ci_t])
acc_approx = val_acc(approx)
print(f"exact SVC (CPU, trained on {n_chk}, fit {t_exact:.0f}s): val acc {acc_exact:.4f}")
print(f"GPU approximation (same {n_chk} images):       val acc {acc_approx:.4f}")